In [ ]:
#PyTorch 版本 以及 Cuda 調用確認
import torch

print(f"PyTorch 版本: {torch.__version__}") # 應該要 >= 2.6.0
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"啟動成功！")
    print(f"顯卡名稱: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 驅動上限: {torch.version.cuda}")

PyTorch 版本: 2.11.0+cu126
CUDA 是否可用: True
啟動成功！
顯卡名稱: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA 驅動上限: 12.6


In [ ]:
#上銀螺桿Embeding工程

import json
import chromadb
from chromadb.utils import embedding_functions

# 1. 讀取 JSON 檔案
with open("final_chunks.json", "r", encoding="utf-8") as f:
    final_chunks = json.load(f)

print(f"成功讀取 {len(final_chunks)} 個切片。")

# 2. 設定 Embedding 模型 (BGE-M3)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 有 NVIDIA 顯卡用 cuda，沒有則改 "cpu"
)

# 3. 初始化本地向量資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# --- [重點修改：確保資料夾乾淨並使用新格式] ---
collection_name = "hiwin_manual"

# 如果 collection 已經存在，先刪除它，確保舊格式(沒 brand 的)不會混進來
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(name=collection_name)
    print(f"已刪除舊的 {collection_name}，準備寫入新格式資料...")

# 建立新的 Collection
collection = client.create_collection(
    name=collection_name,
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)

# 4. 準備資料與「格式化」Metadata
documents = []
new_metadatas = []
ids = []

for i, chunk in enumerate(final_chunks):
    documents.append(chunk['content'])
    
    # 這裡將原本 JSON 裡的 metadata 擴充為「必要項目」格式
    # 如果你的 JSON 裡原本就有 page，可以直接引用 chunk['metadata']['source_page']
    refined_meta = {
        "brand": "HIWIN",                   # 固定標註為上銀
        "category": "Screw",                # 零件大類 (手冊通常是螺桿)
        "data_type": "Manual",              # 資料類型
        "source_file": "hiwin_manual_v1.pdf", # 原始檔案名稱
        "page": chunk.get('metadata', {}).get('source_page', 0) # 嘗試抓取原有的頁碼
    }
    new_metadatas.append(refined_meta)
    ids.append(f"hiwin_m_{i}") # 使用更具辨識度的 ID

# 5. 執行 Embedding 並分批存入資料庫
print(f"開始執行 BGE-M3 Embedding 轉換... (共 {len(documents)} 筆)")

batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    collection.add(
        documents=documents[i:end],
        metadatas=new_metadatas[i:end],
        ids=ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"向量資料庫 '{collection_name}' 建置完成！")
print(f"資料夾路徑：./hiwin_vector_db")

成功讀取 87 個切片。
已刪除舊的 hiwin_manual，準備寫入新格式資料...
開始執行 BGE-M3 Embedding 轉換... (共 87 筆)
已完成: 50/87
已完成: 87/87
向量資料庫 'hiwin_manual' 建置完成！
資料夾路徑：./hiwin_vector_db


In [1]:
#Excel 語意欄位建置

import pandas as pd

# 1. 設定檔案路徑
file_path = R"../data/HIWIN_Specs.xlsx"

def process_hiwin_specs(path):
    # 2. 讀取 ALL 工作表
    # 我們在函數內部讀取，這樣 ExcelWriter 才能正確處理檔案鎖定
    df = pd.read_excel(path, sheet_name="ALL")

    # 定義語意合成邏輯
    def create_full_semantic_text(row):
        series = str(row.get('系列', '')).strip()
        model = str(row.get('型號', '')).strip()
        outer_dia = str(row.get('公稱 外徑', '0'))
        lead = str(row.get('導程', '0'))
        ball_dia = str(row.get('珠徑', '0'))
        pcd = str(row.get('PCD', '0'))
        root_dia = str(row.get('根徑', '0'))
        turns = str(row.get('珠卷數', '0'))
        stiffness = str(row.get('剛性 kfg/umk', '0'))
        dyn_load = str(row.get('動負荷 C (kfg)', '0'))
        stat_load = str(row.get('靜負荷 Co (kfg)', '0'))
        
        text = (
            f"上銀 HIWIN 滾珠螺桿型號 {model} (系列: {series})。 "
            f"幾何規格：公稱外徑 {outer_dia}mm，導程 {lead}mm，珠徑 {ball_dia}mm，"
            f"節圓直徑(PCD) {pcd}mm，根徑 {root_dia}mm，珠卷數為 {turns}。 "
            f"機械性能：剛性達 {stiffness} kfg/umk，"
            f"額定動負荷(C)為 {dyn_load} kgf，靜負荷(Co)為 {stat_load} kgf。"
        )
        
        if series == 'FDC':
            text += " 此型號為雙螺帽設計，具備極高剛性，專為重負荷精密機台開發。"
        elif series in ['FSI', 'FSW']:
            text += " 此型號結構輕巧省空間，適合小型自動化設備或精密儀器。"
        elif series == 'FSV':
            text += " 此型號為標準單螺帽設計，傳動效率優異，是工業自動化最通用的選型。"
        elif series == 'RSI':
            text += " 此型號為旋轉螺帽設計，適合長行程且需高速旋轉螺帽的特殊機構。"
            
        return text

    print("正在將全欄位規格轉換為語意描述...")
    df['semantic_text'] = df.apply(create_full_semantic_text, axis=1)

    # 3. 寫回原檔案的 ALL 工作表
    # 使用 mode='a' (append) 與 if_sheet_exists='replace' 來更新特定分頁
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='ALL', index=False)
    
    print(f"處理完成！語意欄位已更新至 {path} 的 ALL 工作表中。")

# 執行函數
process_hiwin_specs(file_path)

正在將全欄位規格轉換為語意描述...
處理完成！語意欄位已更新至 ../data/HIWIN_Specs.xlsx 的 ALL 工作表中。


In [ ]:
# 上銀螺桿型號表 Embedding 工程 
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

# 1. 初始化 Embedding 模型
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"
)

# 2. 連接資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 強制刷新 Collection (確保舊格式不殘留)
collection_name = "hiwin_specs"
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(name=collection_name)
    print(f"已刪除舊的 {collection_name}，準備寫入新格式規格資料...")

spec_collection = client.create_collection(
    name=collection_name, 
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"} # 指定計算方式
)

# 4. 讀取 Excel
path = R"../data/HIWIN_Specs.xlsx"
df = pd.read_excel(path, sheet_name="ALL")
df = df.fillna("") # 處理空值

# --- [重點優化：Metadata 注入與標準化] ---
documents = df['semantic_text'].tolist()
raw_metadatas = df.drop(columns=['semantic_text']).to_dict('records')

refined_metadatas = []
for i, row in enumerate(raw_metadatas):
    # 建立「必要項目」並整合原始數據
    meta = {
        "brand": "HIWIN",               # 固定注入：品牌
        "category": "Screw",            # 固定注入：分類
        "data_type": "Specification",   # 固定注入：資料類型
        # 將關鍵物理參數提取到頂層（方便 where 過濾），其餘保留
        "dia": float(row.get('公稱 外徑', 0)) if row.get('公稱 外徑') != "" else 0,
        "lead": float(row.get('導程', 0)) if row.get('導程') != "" else 0,
        "model_id": str(row.get('型號', f"unnamed_{i}")),
    }
    # 將原始 Excel 的所有欄位也塞進去，確保資訊不遺失
    meta.update(row) 
    refined_metadatas.append(meta)

# ID 保持唯一性
ids = [f"hiwin_s_{i}_{row.get('型號', i)}" for i, row in enumerate(raw_metadatas)]

# 5. 執行批次存入 (增加 batch_size 處理)
print(f"正在寫入 {len(documents)} 筆資料到 {collection_name}...")
batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    spec_collection.add(
        documents=documents[i:end],
        metadatas=refined_metadatas[i:end],
        ids=ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"成功！目前 {collection_name} 共有 {spec_collection.count()} 筆格式化資料。")

已刪除舊的 hiwin_specs，準備寫入新格式規格資料...
正在寫入 466 筆資料到 hiwin_specs...
已完成: 50/466
已完成: 100/466
已完成: 150/466
已完成: 200/466
已完成: 250/466
已完成: 300/466
已完成: 350/466
已完成: 400/466
已完成: 450/466
已完成: 466/466
成功！目前 hiwin_specs 共有 466 筆格式化資料。


In [3]:
# 銀泰螺桿型號表 Embedding 工程 
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

# 1. 初始化 Embedding 模型
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"
)

# 2. 連接資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 強制刷新 Collection (確保舊格式不殘留)
collection_name = "pmi_specs"
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(name=collection_name)
    print(f"已刪除舊的 {collection_name}，準備寫入新格式規格資料...")

spec_collection = client.create_collection(
    name=collection_name, 
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"} # 指定計算方式
)

# 4. 讀取 Excel
path = R"C:\Users\e11338\Desktop\Feed System GAI\data\PMI_Specs.xlsx"
df = pd.read_excel(path)
df = df.fillna("") # 處理空值

# --- [重點優化：Metadata 注入與標準化] ---
documents = df['semantic_text'].tolist()
raw_metadatas = df.drop(columns=['semantic_text']).to_dict('records')

refined_metadatas = []
for i, row in enumerate(raw_metadatas):
    # 建立「必要項目」並整合原始數據
    meta = {
        "brand": "PMI",               # 固定注入：品牌
        "category": "Screw",            # 固定注入：分類
        "data_type": "Specification",   # 固定注入：資料類型
        "dia": float(row.get('公稱 外徑', 0)) if row.get('公稱 外徑') != "" else 0,
        "lead": float(row.get('導程', 0)) if row.get('導程') != "" else 0,
        "model_id": str(row.get('型號', f"unnamed_{i}")), # 建議用之前處理好的 model_id
    }
    # 將原始 Excel 的所有欄位也塞進去，確保資訊不遺失
    meta.update(row) 
    refined_metadatas.append(meta)

# ID 保持唯一性
ids = [f"pmi_s_{i}_{row.get('型號', i)}" for i, row in enumerate(raw_metadatas)]

# 5. 執行批次存入 (增加 batch_size 處理)
print(f"正在寫入 {len(documents)} 筆資料到 {collection_name}...")
batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    spec_collection.add(
        documents=documents[i:end],
        metadatas=refined_metadatas[i:end],
        ids=ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"成功！目前 {collection_name} 共有 {spec_collection.count()} 筆格式化資料。")

正在寫入 590 筆資料到 pmi_specs...
已完成: 50/590
已完成: 100/590
已完成: 150/590
已完成: 200/590
已完成: 250/590
已完成: 300/590
已完成: 350/590
已完成: 400/590
已完成: 450/590
已完成: 500/590
已完成: 550/590
已完成: 590/590
成功！目前 pmi_specs 共有 590 筆格式化資料。


In [ ]:
#銀泰文本 Embeding 工程
import json
import chromadb
from chromadb.utils import embedding_functions

# 1. 讀取銀泰 JSON 檔案
pmi_json_path = R"C:\Users\e11338\Desktop\Feed System GAI\code\PMI_final_chunks.json" # 請確保檔案名稱正確
with open(pmi_json_path, "r", encoding="utf-8") as f:
    pmi_chunks = json.load(f)

print(f"成功讀取 {len(pmi_chunks)} 個 PMI 切片。")

# 2. 設定 Embedding 模型 (與上銀一致使用 BGE-M3)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  
)

# 3. 初始化本地向量資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# --- [架構優化：檢查並合併 Collection] ---
# 為了名稱對等，我們將 Collection 命名為更通用的 'screw_manuals'
old_name = "hiwin_manual"
new_name = "screw_manuals"

# 如果舊的 hiwin_manual 存在，我們將其改名為 screw_manuals 以符合統一標準
existing_collections = [c.name for c in client.list_collections()]
if old_name in existing_collections and new_name not in existing_collections:
    client.get_collection(old_name).modify(name=new_name)
    print(f"已將 {old_name} 重新命名為 {new_name}")

# 取得 Collection (如果不存在則建立)
collection = client.get_or_create_collection(
    name=new_name,
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)

# 4. 準備銀泰資料與「統一格式」Metadata
documents = []
new_metadatas = []
ids = []

for i, chunk in enumerate(pmi_chunks):
    documents.append(chunk['content'])
    
    # 嚴格遵循你要求的標準化 Metadata 格式
    refined_meta = {
        "brand": "PMI",                      # 固定標註為銀泰
        "category": "Screw",                 # 零件大類
        "data_type": "Manual",               # 資料類型
        "source_file": "pmi_manual_v1.pdf",  # 銀泰檔案名稱
        "page": chunk.get('metadata', {}).get('source_page', 0) 
    }
    
    # 如果原始 JSON 裡還有其他資訊（例如章節），也可以一併存入
    if 'metadata' in chunk:
        for k, v in chunk['metadata'].items():
            if k not in refined_meta: # 避免覆蓋標準欄位
                refined_meta[k] = v

    new_metadatas.append(refined_meta)
    ids.append(f"pmi_m_{i}") # 使用 pmi_m 前綴區隔

# 5. 執行 Embedding 並分批存入資料庫 (與上銀同一個 Collection)
print(f"開始執行 PMI 資料 Embedding... (目標 Collection: {new_name})")

batch_size = 50
for i in range(0, len(documents), batch_size):
    end = i + batch_size
    collection.add(
        documents=documents[i:end],
        metadatas=new_metadatas[i:end],
        ids=ids[i:end]
    )
    print(f"已完成: {min(end, len(documents))}/{len(documents)}")

print(f"向量資料庫合併完成！目前 '{new_name}' 共有 {collection.count()} 筆資料。")
print(f"包含來源：HIWIN (上銀) 與 PMI (銀泰)")

成功讀取 124 個 PMI 切片。


c:\Users\e11338\Desktop\Feed System GAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 40905.24it/s]


已將 hiwin_manual 重新命名為 screw_manuals
開始執行 PMI 資料 Embedding... (目標 Collection: screw_manuals)
已完成: 50/124
已完成: 100/124
已完成: 124/124
向量資料庫合併完成！目前 'screw_manuals' 共有 211 筆資料。
包含來源：HIWIN (上銀) 與 PMI (銀泰)
